<a href="https://colab.research.google.com/github/amulya0612/NASSCOM/blob/main/WEEK%203/U5_Calculus_for_ML_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U5 — Calculus for ML: Lab
How models learn — by following the gradient downhill — derivatives · partial derivatives · the gradient · chain rule · backpropagation · Hessian

Day 2 · Phase B — Mathematical Foundations · Bridge unit (Module 1.3.8)

# objectives
By the end of this lab you will be able to:

Compute derivatives numerically (finite difference) and symbolically (SymPy)

Take partial derivatives and assemble the gradient of a multivariable function

Apply the chain rule by hand and verify it with SymPy

Run a forward and backward pass through a 2-layer network and derive its gradients

Compute a Hessian, and see how a gradient-descent step uses the gradient

# how to use this lab
Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each # YOUR CODE HERE with working code.

Run cells with Shift + Enter. Run the demos before attempting the exercises.

In [ ]:
# Core imports for the whole lab
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. NUMERICAL DERIVATIVE (finite difference)
# -----------------------------------------------------------

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. NUMERICAL DERIVATIVE (finite difference)
# -----------------------------------------------------------

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')

In [ ]:
# -----------------------------------------------------------
# 🔹 1B. SYMBOLIC DERIVATIVE (SymPy)
# -----------------------------------------------------------

expr = x**2
deriv = sp.diff(expr, x)          # differentiate w.r.t. x
print('d/dx (x**2) =', deriv)     # 2*x

# Evaluate the symbolic derivative at x = 3
print('Symbolic f\'(3) =', deriv.subs(x, 3))


# 🧪 LAB EXERCISE 1 — Numerical vs symbolic derivative
For the function g(x) = x**3 + 2*x:

1.Compute the numerical derivative at x = 2 using a finite difference.

2.Compute the symbolic derivative with sp.diff and print it.

3.Evaluate the symbolic derivative at x = 2 and confirm it matches step 1.

In [ ]:
def g(x):
    return x**3 + 2*x

# 1. Numerical derivative at x = 2 (use h = 1e-6)
# YOUR CODE HERE

# 2. Symbolic derivative of x**3 + 2*x
# YOUR CODE HERE

# 3. Evaluate the symbolic derivative at x = 2 and compare
# YOUR CODE HERE

In [ ]:

# -----------------------------------------------------------
# 🔹 2A. PARTIAL DERIVATIVES
# -----------------------------------------------------------

f2 = x**2 + 3*x*y + y**2

# A partial derivative differentiates ONE variable, holding others fixed
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y

In [ ]:
# -----------------------------------------------------------
# 🔹 2B. THE GRADIENT (vector of partials)
# -----------------------------------------------------------

# The gradient stacks every partial derivative into one vector
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

# Evaluate the gradient at the point (x=1, y=2)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction

# 🧪 LAB EXERCISE 2 — Compute a gradient

In [ ]:

h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
# YOUR CODE HERE

# 2. Assemble the gradient list
# YOUR CODE HERE

# 3. Evaluate at (x=2, y=0)  -> hint: .subs({x: 2, y: 0})
# YOUR CODE HERE

# 3. The chain rule

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. CHAIN RULE BY HAND vs SymPy
# -----------------------------------------------------------

# y = sin(x**2) is a composition: outer = sin(u), inner = u = x**2
# Chain rule:  dy/dx = cos(u) * du/dx = cos(x**2) * 2x
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)



In [ ]:

# -----------------------------------------------------------
# 🔹 3B. CHAINING THREE FUNCTIONS
# -----------------------------------------------------------

# y = (3x + 1)**4  -> outer^4, inner (3x+1)
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3



In [ ]:
# y = exp(x**2 + 1);  inner = x**2 + 1, inner' = 2x, outer' = exp(inner)

# 1. By hand (as a SymPy expression)
# YOUR CODE HERE

# 2. With sp.diff
# YOUR CODE HERE

# 3. Confirm they match
# YOUR CODE HERE

# 4. Backpropagation — the chain rule in a network

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. FORWARD PASS
# -----------------------------------------------------------

# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1                 # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))

In [ ]:

# -----------------------------------------------------------
# 🔹 4B. BACKWARD PASS (gradients via the chain rule)
# -----------------------------------------------------------

# Work backwards from the loss, one link at a time
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
dW2  = h.T @ dy                      # d loss / d W2
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')

In [ ]:

# -----------------------------------------------------------
# 🔹 4C. ONE GRADIENT-DESCENT STEP SHOULD LOWER THE LOSS
# -----------------------------------------------------------

lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2

# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

🧪 LAB EXERCISE 4 — Derive the gradients for a 2-layer network


In [ ]:
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
# YOUR CODE HERE

# 2. Backward pass: dy, dWb, dh, dz1, dWa
# YOUR CODE HERE

# 3. One step with lr = 0.05; print loss before and after
# YOUR CODE HERE

5. Hessian & gradient descent

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. THE HESSIAN (matrix of second derivatives = curvature)
# -----------------------------------------------------------

f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)        # [[2, 3], [3, 2]]


In [ ]:
# -----------------------------------------------------------
# 🔹 5B. GRADIENT DESCENT ON A SIMPLE FUNCTION
# -----------------------------------------------------------

# Minimise f(x) = (x - 4)**2, whose minimum is at x = 4.
# Update rule:  x <- x - lr * f'(x),  with f'(x) = 2*(x - 4)
xv = 0.0          # starting point
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)        # the derivative
    xv = xv - lr * grad        # step against the gradient
print('Converged x:', round(xv, 3), ' (true minimum = 4)')



# 🧪 LAB EXERCISE 5 — Hessian + gradient descent

In [ ]:
# 1. Hessian of x**4 + y**2
# YOUR CODE HERE

# 2. Gradient descent to minimise (x - 7)**2
# xv = 0.0; lr = 0.1
# for step in range(20):
#     grad = ...
#     xv = ...
# print('Final x:', xv)
# YOUR CODE HERE